In [1]:
#| default_exp core.optimizers
#| export

import numpy as np
from typing import List, Union, Optional, Dict, Any

# Import Tensor from Module 01 (now with gradient support from Module 06)
from tinytorch.core.tensor import Tensor

# Enable autograd to add gradient tracking to Tensor
# This module depends on Module 06 (Autograd) being available
from tinytorch.core.autograd import enable_autograd
enable_autograd()

# Constants for optimizer defaults
DEFAULT_LEARNING_RATE_SGD = 0.01  # Default learning rate for SGD
DEFAULT_LEARNING_RATE_ADAM = 0.001  # Default learning rate for Adam/AdamW
DEFAULT_MOMENTUM = 0.9  # Default momentum for SGD
DEFAULT_BETA1 = 0.9  # First moment decay rate for Adam
DEFAULT_BETA2 = 0.999  # Second moment decay rate for Adam
DEFAULT_EPS = 1e-8  # Small epsilon for numerical stability in Adam
DEFAULT_WEIGHT_DECAY_ADAMW = 0.01  # Default weight decay for AdamW

In [5]:
#| export
class Optimizer:
    """
    Base class for all optimizers.

    This class defines the common interface that all optimizers must implement:
    - zero_grad(): Clear gradients from parameters
    - step(): Update parameters based on gradients
    """

    def __init__(self, params: List[Tensor]):
        """
        Initialize optimizer with parameters to optimize.

        TODO: Set up the parameter list for optimization

        APPROACH:
        1. Store parameters as a list for iteration
        2. Validate that all parameters require gradients
        3. Initialize step counter for algorithms that need it

        EXAMPLE:
        >>> linear = Linear(784, 128)
        >>> optimizer = SGD(linear.parameters(), lr=0.01)

        HINT: Store parameters for iteration during optimization steps
        """
        ### BEGIN SOLUTION
        if not isinstance(params, list):
            params = list(params)

        self.params = params

        for param in self.params:
            if isinstance(param, Tensor):
                param.requires_grad = True
                param.grad = None
        self.step_count = 0
        ### END SOLUTION

    def zero_grad(self):
        """
        Clear gradients from all parameters.

        TODO: Reset all parameter gradients to None

        APPROACH:
        1. Iterate through all parameters
        2. Set each parameter's grad to None

        EXAMPLE:
        >>> optimizer.zero_grad()  # Clears all gradients
        >>> assert param.grad is None for param in optimizer.params

        WHY: Gradients accumulate by default, so we need to clear them between batches
        """
        ### BEGIN SOLUTION
        for param in self.params:
            param.grad = None
        ### END SOLUTION

    def step(self):
        """
        Update parameters based on gradients.

        This is abstract - each optimizer implements its own update rule.
        """
        raise NotImplementedError(
            f"Abstract method step() not implemented\n"
            f"  ❌ {self.__class__.__name__} inherits from Optimizer but doesn't define step()\n"
            f"  💡 Each optimizer must implement its own update rule (SGD, Adam, etc.)\n"
            f"  🔧 Override step() in your optimizer subclass:\n"
            f"      def step(self):\n"
            f"          for param in self.params:\n"
            f"              if param.grad is not None:\n"
            f"                  param.data -= self.lr * param.grad.data"
        )

In [6]:
#| export
class _ExtractGradientMixin:
    """Mixin added to Optimizer for gradient extraction."""
    def _extract_gradient(self, param: Tensor) -> np.ndarray:
        """
        Extract gradient data as a NumPy array from a parameter.

        Gradients can be stored as either a Tensor (with .data attribute)
        or as a raw NumPy array (from autograd). This helper normalizes
        both cases to a plain NumPy array for optimizer math.

        TODO: Return the gradient's underlying NumPy array

        APPROACH:
        1. Get param.grad
        2. If it's a Tensor, return its .data attribute
        3. If it's already a NumPy array, return it directly

        EXAMPLE:
        >>> param = Tensor([1.0, 2.0], requires_grad=True)
        >>> param.grad = Tensor([0.1, 0.2])
        >>> optimizer._extract_gradient(param)
        array([0.1, 0.2])

        HINT: Use isinstance(grad, Tensor) to check the type
        """
        ### BEGIN SOLUTION
        grad = param.grad
        if isinstance(grad, Tensor):
            return grad.data
        else:
            return grad
        ### END SOLUTION

# Attach _extract_gradient to Optimizer so all subclasses inherit it
Optimizer._extract_gradient = _ExtractGradientMixin._extract_gradient

In [7]:
def test_unit_extract_gradient():
    """🧪 Test _extract_gradient handles Tensor and ndarray gradients."""
    print("🧪 Unit Test: Gradient Extraction...")

    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = Optimizer([param])

    # Case 1: Gradient is a Tensor
    param.grad = Tensor([0.1, 0.2])
    grad_data = optimizer._extract_gradient(param)
    assert isinstance(grad_data, np.ndarray), "Should return ndarray from Tensor grad"
    assert np.allclose(grad_data, [0.1, 0.2])

    # Case 2: Gradient is a raw NumPy array
    param.grad = np.array([0.3, 0.4])
    grad_data = optimizer._extract_gradient(param)
    assert isinstance(grad_data, np.ndarray), "Should return ndarray from ndarray grad"
    assert np.allclose(grad_data, [0.3, 0.4])

    # Case 3: Multi-dimensional gradient
    param_2d = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
    opt_2d = Optimizer([param_2d])
    param_2d.grad = Tensor([[0.1, 0.2], [0.3, 0.4]])
    grad_data_2d = opt_2d._extract_gradient(param_2d)
    assert grad_data_2d.shape == (2, 2)
    assert np.allclose(grad_data_2d, [[0.1, 0.2], [0.3, 0.4]])

    print("✅ Gradient extraction works correctly!")

if __name__ == "__main__":
    test_unit_extract_gradient()

🧪 Unit Test: Gradient Extraction...
✅ Gradient extraction works correctly!


In [8]:
def test_unit_optimizer_base():
    """🧪 Test base Optimizer functionality."""
    print("🧪 Unit Test: Base Optimizer...")

    # Create test parameters
    param1 = Tensor([1.0, 2.0], requires_grad=True)
    param2 = Tensor([[3.0, 4.0], [5.0, 6.0]], requires_grad=True)

    # Create optimizer first (optimizer.__init__ resets grad to None)
    optimizer = Optimizer([param1, param2])

    # Test parameter storage
    assert len(optimizer.params) == 2
    assert optimizer.params[0] is param1
    assert optimizer.params[1] is param2
    assert optimizer.step_count == 0

    # Add gradients AFTER creating optimizer to test zero_grad properly
    param1.grad = Tensor([0.1, 0.2])
    param2.grad = Tensor([[0.3, 0.4], [0.5, 0.6]])

    # Test zero_grad
    optimizer.zero_grad()
    assert param1.grad is None
    assert param2.grad is None

    # Test that optimizer accepts any tensor (no validation required)
    # Gradient tracking is handled by the autograd module
    regular_param = Tensor([1.0])
    opt = Optimizer([regular_param])
    assert len(opt.params) == 1

    print("✅ Base Optimizer works correctly!")

if __name__ == "__main__":
    test_unit_optimizer_base()

🧪 Unit Test: Base Optimizer...
✅ Base Optimizer works correctly!


In [10]:
#| export
class SGD(Optimizer):
    """
    Stochastic Gradient Descent with momentum.

    SGD is the foundational optimization algorithm that moves parameters
    in the direction opposite to gradients. With momentum, it remembers
    previous updates to reduce oscillations and accelerate convergence.
    """

    def __init__(self, params: List[Tensor], lr: float = DEFAULT_LEARNING_RATE_SGD, momentum: float = 0.0, weight_decay: float = 0.0):
        """
        Initialize SGD optimizer.

        TODO: Set up SGD with momentum and weight decay

        APPROACH:
        1. Call parent constructor to set up parameters
        2. Store learning rate, momentum, and weight decay
        3. Initialize momentum buffers for each parameter

        EXAMPLE:
        >>> optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

        HINTS:
        - Momentum buffers should be initialized as None
        - They'll be created lazily on first step
        """
        ### BEGIN SOLUTION
        super().__init__(params)

        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay

        # Initialize momentum buffers (created lazily)
        self.momentum_buffers = [None for _ in self.params]
        ### END SOLUTION

    def has_momentum(self) -> bool:
        """
        Check if this optimizer uses momentum.

        This explicit API method replaces the need for hasattr() checks
        in checkpointing code (Module 08).

        Returns:
            bool: True if momentum is enabled (momentum > 0), False otherwise

        EXAMPLE:
            >>> optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> optimizer.has_momentum()
            True
        """
        return self.momentum > 0

    def get_momentum_state(self) -> Optional[List]:
        """
        Get momentum buffers for checkpointing.

        This explicit API method provides safe access to momentum buffers
        without using hasattr(), making the API contract clear.

        Returns:
            Optional[List]: List of momentum buffers if momentum is enabled,
                          None otherwise

        EXAMPLE:
            >>> optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> optimizer.step()  # Initialize buffers
            >>> state = optimizer.get_momentum_state()
            >>> # Later: optimizer.set_momentum_state(state)
        """
        if not self.has_momentum():
            return None
        return [buf.copy() if buf is not None else None
                for buf in self.momentum_buffers]

    def set_momentum_state(self, state: Optional[List]) -> None:
        """
        Restore momentum buffers from checkpointing.

        This explicit API method provides safe restoration of momentum state
        without using hasattr().

        Args:
            state: List of momentum buffers or None

        EXAMPLE:
            >>> optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> state = optimizer.get_momentum_state()
            >>> # Training interruption...
            >>> new_optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> new_optimizer.set_momentum_state(state)
        """
        if state is None or not self.has_momentum():
            return

        if len(state) != len(self.momentum_buffers):
            raise ValueError(
                f"Momentum state length mismatch\n"
                f"  ❌ State has {len(state)} buffers, but optimizer has {len(self.momentum_buffers)} parameters\n"
                f"  💡 Checkpoint was saved with a different model architecture or parameter count\n"
                f"  🔧 Ensure you're loading state into an optimizer with the same number of parameters:\n"
                f"      # Check parameter counts match before restoring\n"
                f"      assert len(saved_state) == len(optimizer.params)"
            )

        for i, buf in enumerate(state):
            if buf is not None:
                self.momentum_buffers[i] = buf.copy()

    def step(self):
        """
        Perform SGD update step with momentum.

        TODO: Implement SGD parameter update by composing helpers

        APPROACH:
        1. For each parameter with gradients:
           a. Extract gradient using self._extract_gradient(param)
           b. Apply weight decay if specified
           c. Update momentum buffer
           d. Update parameter using momentum

        FORMULA:
        - With weight decay: grad = grad + weight_decay * param
        - Momentum: v = momentum * v_prev + grad
        - Update: param = param - lr * v

        HINTS:
        - Skip parameters without gradients
        - Use self._extract_gradient() from the base class
        - Initialize momentum buffers on first use
        """
        ### BEGIN SOLUTION
        for i, param in enumerate(self.params):
            if param.grad is None:
                continue

            # extract gradient using shared helper
            grad_data = self._extract_gradient(param)

            # apply weight decay
            if self.weight_decay != 0:
                grad_data = grad_data + self.weight_decay * param.data

            # update momentum buffer
            if self.momentum != 0:
                if self.momentum_buffers[i] is None:
                    # initialize momentum buffer
                    self.momentum_buffers[i] = np.zeros_like(param.data)

                # update momentum: v = momentum * v_prev + grad
                self.momentum_buffers[i] = self.momentum * self.momentum_buffers[i] + grad_data
                grad_data = self.momentum_buffers[i]

            # update parameter: param = param - lr * grad
            param.data = param.data - self.lr * grad_data

        # increment step counter
        self.step_count += 1
        ### END SOLUTION

In [11]:
def test_unit_sgd_optimizer():
    """🧪 Test SGD optimizer implementation."""
    print("🧪 Unit Test: SGD Optimizer...")

    # Test basic SGD without momentum
    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = SGD([param], lr=0.1)
    # Set gradient AFTER creating optimizer (optimizer.__init__ resets grad to None)
    param.grad = Tensor([0.1, 0.2])
    original_data = param.data.copy()

    optimizer.step()

    # Expected: param = param - lr * grad = [1.0, 2.0] - 0.1 * [0.1, 0.2] = [0.99, 1.98]
    expected = original_data - 0.1 * np.array([0.1, 0.2])
    assert np.allclose(param.data, expected)
    assert optimizer.step_count == 1

    # Test SGD with momentum
    param2 = Tensor([1.0, 2.0], requires_grad=True)
    optimizer_momentum = SGD([param2], lr=0.1, momentum=0.9)
    # Set gradient AFTER creating optimizer
    param2.grad = Tensor([0.1, 0.2])

    # First step: v = 0.9 * 0 + [0.1, 0.2] = [0.1, 0.2]
    optimizer_momentum.step()
    expected_first = np.array([1.0, 2.0]) - 0.1 * np.array([0.1, 0.2])
    assert np.allclose(param2.data, expected_first)

    # Second step with same gradient
    param2.grad = Tensor([0.1, 0.2])
    optimizer_momentum.step()
    # v = 0.9 * [0.1, 0.2] + [0.1, 0.2] = [0.19, 0.38]
    expected_momentum = np.array([0.19, 0.38])
    expected_second = expected_first - 0.1 * expected_momentum
    assert np.allclose(param2.data, expected_second, rtol=1e-5)

    # Test weight decay
    param3 = Tensor([1.0, 2.0], requires_grad=True)
    optimizer_wd = SGD([param3], lr=0.1, weight_decay=0.01)
    # Set gradient AFTER creating optimizer
    param3.grad = Tensor([0.1, 0.2])
    optimizer_wd.step()

    # grad_with_decay = [0.1, 0.2] + 0.01 * [1.0, 2.0] = [0.11, 0.22]
    expected_wd = np.array([1.0, 2.0]) - 0.1 * np.array([0.11, 0.22])
    assert np.allclose(param3.data, expected_wd)

    print("✅ SGD optimizer works correctly!")

if __name__ == "__main__":
    test_unit_sgd_optimizer()

🧪 Unit Test: SGD Optimizer...
✅ SGD optimizer works correctly!


In [15]:
#| export
class Adam(Optimizer):
    """
    Adam optimizer with adaptive learning rates.

    Adam computes individual adaptive learning rates for different parameters
    from estimates of first and second moments of the gradients.
    This makes it effective for problems with sparse gradients or noisy data.
    """

    def __init__(self, params: List[Tensor], lr: float = DEFAULT_LEARNING_RATE_ADAM, betas: tuple = (DEFAULT_BETA1, DEFAULT_BETA2), eps: float = DEFAULT_EPS, weight_decay: float = 0.0):
        """
        Initialize Adam optimizer.

        TODO: Set up Adam with adaptive learning rates

        APPROACH:
        1. Call parent constructor
        2. Store hyperparameters (lr, betas, eps, weight_decay)
        3. Initialize first and second moment buffers

        PARAMETERS:
        - lr: Learning rate (default: 0.001)
        - betas: Coefficients for computing running averages (default: (0.9, 0.999))
        - eps: Small constant for numerical stability (default: 1e-8)
        - weight_decay: L2 penalty coefficient (default: 0.0)

        EXAMPLE:
        >>> optimizer = Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))
        """
        ### BEGIN SOLUTION
        super().__init__(params)

        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay

        # initialize moment buffers (created lazily)
        self.m_buffers = [None for _ in self.params] # first moment (mean)
        self.v_buffers = [None for _ in self.params] # second moment (variance)
        ### END SOLUTION

In [18]:
#| export
class _AdamUpdateMomentsMixin:
    """Mixin added to Adam for moment updates."""
    def _update_moments(self, i: int, grad_data: np.ndarray) -> tuple:
        """
        Update first and second moment estimates with bias correction.

        Computes the exponential moving averages of the gradient (first moment)
        and the squared gradient (second moment), then applies bias correction
        to counteract the zero-initialization bias in early training steps.

        TODO: Update moment buffers and return bias-corrected estimates

        APPROACH:
        1. Initialize m and v buffers to zeros if this is the first call
        2. Update first moment: m = beta1 * m + (1 - beta1) * grad
        3. Update second moment: v = beta2 * v + (1 - beta2) * grad^2
        4. Compute bias corrections using step_count
        5. Return bias-corrected m_hat and v_hat

        EXAMPLE:
        >>> m_hat, v_hat = self._update_moments(0, np.array([0.1, 0.2]))
        >>> # m_hat ≈ grad (after bias correction at step 1)
        >>> # v_hat ≈ grad^2 (after bias correction at step 1)

        HINTS:
        - Use self.step_count (already incremented before this call)
        - Bias correction denominators: (1 - beta^t) approach 1 as t grows
        """
        ### BEGIN SOLUTION
        # Initialize buffers if needed
        if self.m_buffers[i] is None:
            self.m_buffers[i] = np.zeros_like(grad_data)
            self.v_buffers[i] = np.zeros_like(grad_data)

        # update biased first moment estimate
        self.m_buffers[i] = self.beta1 * self.m_buffers[i] + (1 - self.beta1) * grad_data

        # update biased second moment estimate
        self.v_buffers[i] = self.beta2 * self.v_buffers[i] + (1 - self.beta2) * (grad_data ** 2)

        # compute bias correction
        bias_correction1 = 1 - self.beta1 ** self.step_count
        bias_correction2 = 1 - self.beta2 ** self.step_count

        # compute bias-corrected moments
        m_hat = self.m_buffers[i] / bias_correction1
        v_hat = self.v_buffers[i] / bias_correction2

        return m_hat, v_hat
        ### END SOLUTION

# Attach _update_moments to Adam
Adam._update_moments = _AdamUpdateMomentsMixin._update_moments

In [19]:
def test_unit_adam_update_moments():
    """🧪 Test Adam _update_moments computes correct EMA and bias correction."""
    print("🧪 Unit Test: Adam Moment Updates...")

    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = Adam([param], lr=0.01, betas=(0.9, 0.999), eps=1e-8)

    grad = np.array([0.1, 0.2])

    # Simulate step 1 (step_count must be set before calling _update_moments)
    optimizer.step_count = 1

    m_hat, v_hat = optimizer._update_moments(0, grad)

    # Manual calculation for step 1:
    # m = 0.9 * 0 + 0.1 * [0.1, 0.2] = [0.01, 0.02]
    # v = 0.999 * 0 + 0.001 * [0.01, 0.04] = [0.00001, 0.00004]
    # bias_correction1 = 1 - 0.9^1 = 0.1
    # bias_correction2 = 1 - 0.999^1 = 0.001
    # m_hat = [0.01, 0.02] / 0.1 = [0.1, 0.2]  (= grad, as expected!)
    # v_hat = [0.00001, 0.00004] / 0.001 = [0.01, 0.04]  (= grad^2)

    assert np.allclose(m_hat, grad), f"m_hat should equal grad at step 1, got {m_hat}"
    assert np.allclose(v_hat, grad ** 2), f"v_hat should equal grad^2 at step 1, got {v_hat}"

    # Step 2 with same gradient
    optimizer.step_count = 2
    m_hat2, v_hat2 = optimizer._update_moments(0, grad)

    # Moments should still be close to grad (converging to true mean)
    assert m_hat2 is not None
    assert v_hat2 is not None
    # Buffers should be updated in-place
    assert optimizer.m_buffers[0] is not None
    assert optimizer.v_buffers[0] is not None

    print("✅ Adam moment updates work correctly!")

if __name__ == "__main__":
    test_unit_adam_update_moments()

🧪 Unit Test: Adam Moment Updates...
✅ Adam moment updates work correctly!


In [ ]:
#| export
class _AdamStepMixin:
    """Mixin added to Adam for step method."""
    def step(self):
        """
        Perform Adam update step by composing helpers.

        TODO: Implement Adam parameter update using _extract_gradient and _update_moments

        APPROACH:
        1. Increment step_count (needed for bias correction)
        2. For each parameter with gradients:
           a. Extract gradient with self._extract_gradient(param)
           b. Apply weight decay to gradient if specified
           c. Update moments with self._update_moments(i, grad_data)
           d. Update parameter: param -= lr * m_hat / (sqrt(v_hat) + eps)

        FORMULAS:
        - θ_t = θ_{t-1} - lr * m̂_t / (√v̂_t + ε)

        HINTS:
        - Increment step_count FIRST (before the loop)
        - _update_moments returns (m_hat, v_hat) tuple
        - Weight decay modifies grad_data before moment update
        """
        ### BEGIN SOLUTION
        # Increment step counter first (needed for bias correction)
        self.step_count += 1

        for i, param in enumerate(self.params):
            if param.grad is None:
                continue

            # extract gradient using shared helper
            grad_data = self._extract_gradient(param)

            # apply weight decay
            if self.weight_decay != 0:
                grad_data = grad_data + self.weight_decay * param.data
            
            # update moments and get bias-corrected estimates
            m_hat, v_hat = self._update_moments(i, grad_data)

            # update parameter
            param.data = param.data - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        ### END SOLUTION

# Attach step to Adam
Adam.step = _AdamStepMixin.step